In [1]:
#HDDM N-back script using bayesian MCMC to converge on set of parameters. We use the baseline model from https://www.sciencedirect.com/science/article/pii/S2451902222000787 and make variations for model validation.

# Load packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import hddm
from joblib import Parallel, delayed

In [ ]:
import hddm
import numpy as np
import pandas as pd
import arviz as az

def fit_subject_with_convergence(data, chains=4):
    """
    Fits HDDMStimCoding for each subject using multiple chains and computes R-hat and DIC automatically.
    """
    rhat_results_list = []
    dic_results_list = []
    trace_results_list = []

    for subj_idx in np.unique(data['subj_idx']):
        subj_data = data[data['subj_idx'] == subj_idx]

        # Store traces from all chains for this subject
        chain_traces = []

        print(f"\nFitting subject {subj_idx} with {chains} chains...")

        for c in range(chains):
            print(f"  Chain {c+1}")
            m = hddm.HDDMStimCoding(
                subj_data,
                stim_col='stim',
                split_param='v',
                drift_criterion=False,
                bias=False,
                include=['st', 'sv'],
                p_outlier=0.01,  # Enables robust modeling of outliers (e.g., imputed missed trials)
                depends_on={'v': 'condition', 'a': 'condition', 't': 'condition', 'z': 'condition', 'st': 'condition', 'sv': 'condition'}
            )
            m.sample(4000, burn=1000)
            chain_traces.append(m.get_traces())

        # Stack traces into a dict for arviz
        param_names = chain_traces[0].columns
        trace_dict = {
            param: [trace[param].values for trace in chain_traces]
            for param in param_names
        }

        # Convert to arviz InferenceData and calculate R-hat
        idata = az.from_dict(posterior=trace_dict)
        rhat = az.rhat(idata)

        # Get DIC from the model directly
        dic_value = m.dic

        # Save R-hat values
        for param, value in rhat.items():
            rhat_results_list.append({
                'subj_idx': subj_idx,
                'param': param,
                'rhat': float(value)
            })

        # Save DIC value
        dic_results_list.append({
            'subj_idx': subj_idx,
            'dic': dic_value
        })

        # Average trace values (from first chain or across chains)
        mean_traces = chain_traces[0].mean()
        for param in mean_traces.index:
            trace_results_list.append({
                'subj_idx': subj_idx,
                'param': param,
                'mean_trace_value': mean_traces[param]
            })

    return pd.DataFrame(trace_results_list), pd.DataFrame(rhat_results_list), pd.DataFrame(dic_results_list)


# Load data
data = pd.read_csv(r'DDM_Full.csv')

# Handle NaN RTs: treat as missed responses with long RTs and incorrect
data['rt'] = data['rt'].fillna(5.0)
data['correct'] = np.where(data['rt'] == 5.0, 0, data['correct'])
data['response'] = data['response'].fillna(0)  # Fill response to something valid (model needs integers)

# Run model and collect results
df_traces, df_rhat, df_dic = fit_subject_with_convergence(data)

# Save results
df_traces.to_csv('hddm_traces_4.csv', index=False)
df_rhat.to_csv('hddm_rhat_4.csv', index=False)
df_dic.to_csv('hddm_dic_4.csv', index=False)


In [ ]:
#Getting DIC value for combined model

import hddm
import numpy as np
import pandas as pd
import arviz as az

def fit_combined_model(data, chains=4):
    """
    Fits a hierarchical HDDM model for all subjects combined using multiple chains and computes R-hat and DIC automatically.
    """
    rhat_results_list = []
    dic_results_list = []  # List to store DIC results
    trace_results_list = []

    # Combine data from all subjects for the hierarchical model
    print("\nFitting combined hierarchical model...")

    # Fit the combined hierarchical model
    chain_traces = []

    for c in range(chains):
        print(f"  Chain {c+1}")
        m = hddm.HDDMStimCoding(
            data,
            stim_col='stim',
            split_param='z',
            drift_criterion=False, 
            bias=True,
            p_outlier=0.05,
            #include=('sv', 'st'),
             depends_on={'v': 'condition', 'a': 'condition', 't': 'condition', 'sv': 'condition', 'sv': 'condition', 'st': 'condition', 'sz': 'condition', 'z': 'condition'}
        )
        m.sample(2000, burn=1000)
        chain_traces.append(m.get_traces())

    # Stack traces into a dict for arviz
    param_names = chain_traces[0].columns
    trace_dict = {
        param: [trace[param].values for trace in chain_traces]
        for param in param_names
    }

    # Convert to arviz InferenceData and calculate R-hat
    idata = az.from_dict(posterior=trace_dict)

    # Calculate R-hat
    rhat = az.rhat(idata)

    # Get DIC from the combined model directly
    dic_value = m.dic  # This is for the combined model

    # Save R-hat values
    for param, value in rhat.items():
        rhat_results_list.append({
            'param': param,
            'rhat': float(value)
        })

    # Save combined DIC value
    dic_results_list.append({
        'dic': dic_value
    })

    # Optional: average trace values (just from chain 0 here, or average across chains)
    mean_traces = chain_traces[0].mean()  # You can replace with across-chain mean if you prefer
    for param in mean_traces.index:
        trace_results_list.append({
            'param': param,
            'mean_trace_value': mean_traces[param]
        })

    return pd.DataFrame(trace_results_list), pd.DataFrame(rhat_results_list), pd.DataFrame(dic_results_list)


# Load and clean data
# Load and clean data
data = pd.read_csv(r'DDM_Full.csv')

# Handle NaN RTs: treat as missed responses with long RTs and incorrect
data['rt'] = data['rt'].fillna(5.0)
data['correct'] = np.where(data['rt'] == 5.0, 0, data['correct'])
data['response'] = data['response'].fillna(0)  # Fill response to something valid (model needs integers)

# Run model and collect results
df_traces, df_rhat, df_dic = fit_combined_model(data)

# Save results
df_traces.to_csv('hddm_traces_combined_sim.csv', index=False)
df_rhat.to_csv('hddm_rhat_combined_sim.csv', index=False)
df_dic.to_csv('hddm_dic_combined_sim.csv', index=False)

In [48]:
import pandas as pd

# Step 1: Load your CSV
hddm_traces = pd.read_csv("hddm_traces_4.csv")

# Step 2: Extract 'param_type' and 'condition' from the updated 'param' strings
hddm_traces[['param_type', 'condition']] = hddm_traces['param'].str.extract(r'([a-zA-Z]+)\((\d+-back)\)')

# Step 3: Pivot the table
hddm_pivot = hddm_traces.pivot_table(
    index=['subj_idx', 'condition'],
    columns='param_type',
    values='mean_trace_value'
).reset_index()

# Step 4: Clean column names
hddm_pivot.columns.name = None  # Remove param_type label

hddm_pivot.to_csv('hddm_traces_pivot.csv', index=False)


In [ ]:
import pandas as pd

# Step 1: Load CSV
hddm_traces = pd.read_csv("hddm_traces_combined_sim.csv")

# Step 2: Filter only subject-level parameter rows
subj_rows = hddm_traces[hddm_traces['param'].str.contains(r'_subj\(\d+-back\)\.P\d+', regex=True)]

# Step 3: Extract param_type, condition, and subject index
subj_rows[['param_type', 'condition', 'subj_idx']] = subj_rows['param'].str.extract(
    r'([a-zA-Z]+)_subj\((\d+-back)\)\.(P\d+)'  # e.g. a_subj(1-back).P002
)

# Step 4: Pivot table
hddm_pivot = subj_rows.pivot_table(
    index=['subj_idx', 'condition'],
    columns='param_type',
    values='mean_trace_value'
).reset_index()

# Step 5: Clean column names
hddm_pivot.columns.name = None

# Step 6: Save to CSV
hddm_pivot.to_csv('hddm_traces_pivot.csv', index=False)

In [ ]:
import pandas as pd
import numpy as np
import hddm

def get_choice(row):
    if row.stim == 'present':
        return int(row.response == 1)
    elif row.stim == 'absent':
        return int(row.response == 0)

def simulate_nback_data(a, v, t, sv, st, cond_label, nr_trials_per_stim=20, subj_noise=0.05):
    parameters = {'a': a, 'v': v, 't': t, 'sv' : sv, 'st' : st}
    
    # Simulate 'present' stimuli
    df_sim1, _ = hddm.generate.gen_rand_data(params=parameters, size=nr_trials_per_stim, subjs=59, subj_noise=subj_noise)
    df_sim1['stim'] = 'present'
    
    # Simulate 'absent' stimuli
    df_sim2, _ = hddm.generate.gen_rand_data(params=parameters, size=nr_trials_per_stim, subjs=59, subj_noise=subj_noise)
    df_sim2['stim'] = 'absent'
    
    # Combine both conditions
    df_sim = pd.concat([df_sim1, df_sim2], ignore_index=True)
    
    # Apply response logic
    df_sim['bias_response'] = df_sim.apply(get_choice, axis=1)
    df_sim['correct'] = df_sim['response'].astype(int)
    df_sim['response'] = df_sim['bias_response'].astype(int)
    
    # Final stimulus coding
    df_sim['stimulus'] = ((df_sim['response'] == 1) & (df_sim['correct'] == 1) |
                          (df_sim['response'] == 0) & (df_sim['correct'] == 0)).astype(int)
    
    # Clean up
    df_sim.drop(columns=['bias_response'], inplace=True)
    df_sim['condition'] = cond_label
    
    return df_sim

# Load your subject-level parameters
hddm_traces_pivot = pd.read_csv("hddm_traces_pivot.csv")

# Initialize full simulated dataset
df_all_simulated = pd.DataFrame()

# Loop through each subject-condition row
for _, row in hddm_traces_pivot.iterrows():
    a = row['a']
    v = row['v']
    t = row['t']
    sv = row['sv']
    st = row['st']
    cond_label = row['condition']
    subj_idx = row['subj_idx']

    df_sim = simulate_nback_data(a, v, t, sv, st, cond_label, nr_trials_per_stim=20)
    df_sim['subj_idx'] = subj_idx

    df_all_simulated = pd.concat([df_all_simulated, df_sim], ignore_index=True)

# Preview and save
print(df_all_simulated['condition'].value_counts())
print(df_all_simulated.head())

df_all_simulated.to_csv("simulated_nback_data.csv", index=False)


In [ ]:
def simulate_ddm_data(true_param_dict, stim_drift_dict, nr_trials_per_cond=100, n_subj=100, subj_noise=0.05):
    """
    Simulate DDM data with drift rate depending on stimulus within each condition.

    Args:
        true_param_dict: dict mapping condition name to a dict of base DDM parameters (excluding drift).
        stim_drift_dict: dict mapping stimulus value (0 or 1) to drift rate per condition.
                         Example:
                         {
                             '0-back': {0: -0.5, 1: 0.5},
                             '1-back': {0: -0.4, 1: 0.4},
                             ...
                         }
        ...
    """
    all_data = []

    for subj in range(n_subj):
        subj_id = f'P{subj+1:03d}'

        for cond, base_params in true_param_dict.items():
            cond_data = []

            for stim_val in [0, 1]:
                n_trials = int(nr_trials_per_cond * (0.25 if stim_val == 1 else 0.75))
                params = base_params.copy()
                params['v'] = stim_drift_dict[cond][stim_val]

                df_stim, _ = hddm.generate.gen_rand_data(params=params, size=n_trials,
                                                         subjs=1, subj_noise=subj_noise)
                df_stim['stim'] = stim_val
                df_stim['condition'] = cond
                df_stim['subj_idx'] = subj_id
                df_stim['correct'] = (df_stim['response'] == stim_val).astype(int)

                cond_data.append(df_stim)

            all_data.append(pd.concat(cond_data, ignore_index=True))

    return pd.concat(all_data, ignore_index=True)

true_params_by_condition = {
    '0-back': {'a': 0.5, 't': 0.15, 'st': 0.2, 'sv': 1},
    '1-back': {'a': 0.5, 't': 0.15, 'st': 0.2, 'sv': 1},
    '2-back': {'a': 0.5, 't': 0.15, 'st': 0.2, 'sv': 1},
    '3-back': {'a': 0.5, 't': 0.15, 'st': 0.2, 'sv': 1},
}

stim_drift_by_condition = {
    '0-back': {0: -0.5, 1: 0.8},
    '1-back': {0: -0.5, 1: 0.8},
    '2-back': {0: -0.5, 1: 0.8},
    '3-back': {0: -0.5, 1: 0.8},
}

# Simulate data
df_simulated = simulate_ddm_data(true_params_by_condition, stim_drift_by_condition,
                                 nr_trials_per_cond=100, n_subj=500, subj_noise=0.05)
# Save to CSV
df_simulated.to_csv("parameter_recovery_sim_multi_cond.csv", index=False)

# Print head of the simulated dataframe to check the data
print(df_simulated.head())

In [ ]:
import numpy as np
import pandas as pd
import hddm

def simulate_ddm_grid(a=0.5, st=0.2, sv=1.0, sz=0.0, n_trials=10000, n_bins=10, seed=42):
    np.random.seed(seed)
    
    Ter_range = np.linspace(0.2, 0.7, n_bins)
    v_range = np.linspace(0.5, 6.0, n_bins)
    
    results = []

    for Ter in Ter_range:
        for v in v_range:
            params = {'a': a, 'v': v, 't': Ter, 'st': st, 'sv': sv, 'sz': sz}
            try:
                sim_data, _ = hddm.generate.gen_rand_data(params=params, size=n_trials)
                acc = np.mean(sim_data['response'] == 1)  # 1 = correct
                rt = np.mean(sim_data['rt'])
            except Exception as e:
                print(f"Simulation failed for v={v:.2f}, Ter={Ter:.2f}: {e}")
                acc, rt = np.nan, np.nan
            results.append({'Ter': Ter, 'v': v, 'Accuracy': acc, 'RT': rt})
    
    df = pd.DataFrame(results)
    df.to_csv("ddm_heatmap_data.csv", index=False)
    return df

# Run it
df_result = simulate_ddm_grid()
print(df_result.head())


In [ ]:
import pandas as pd
import re

# Step 1: Load the raw traces
hddm_traces = pd.read_csv("hddm_traces_combined.csv")

# Step 2: Extract parameter type (a, v, t) and subject index (e.g., .0, .1)
def extract_param_info(param_str):
    match = re.match(r'([a-zA-Z]+)_subj\(simulated\)\.(\d+)', param_str)
    if match:
        param_type, subj_idx = match.groups()
        return param_type, int(subj_idx)
    else:
        return None, None

hddm_traces[['param_type', 'subj_idx']] = hddm_traces['param'].apply(
    lambda x: pd.Series(extract_param_info(x))
)

# Step 3: Drop rows where extraction failed (non-subject-level rows like a(simulated), a_std, etc.)
hddm_traces = hddm_traces.dropna(subset=['param_type', 'subj_idx'])

# Step 4: Pivot to wide format
hddm_pivot = hddm_traces.pivot_table(
    index='subj_idx',
    columns='param_type',
    values='mean_trace_value'
).reset_index()

# Step 5: Optional — sort by subject
hddm_pivot = hddm_pivot.sort_values(by='subj_idx')

# Step 6: Save to CSV
hddm_pivot.to_csv("hddm_traces_pivot.csv", index=False)

# Preview
print(hddm_pivot.head())